<a href="https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SamarthD07/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
question = """
Research question:
Which pre-March search and engagement signals are associated with stronger
March page-performance outcomes, and can these signals support a transparent
ranking of pages for refresh or review?

Decision supported:
The analysis is intended to help a content/SEO team prioritize which pages
deserve review first when review capacity is limited. The ranking is decision
support, not a claim that refreshing a page will cause better performance.
"""

print(question)


Research question:
Which pre-March search and engagement signals are associated with stronger
March page-performance outcomes, and can these signals support a transparent
ranking of pages for refresh or review?

Decision supported:
The analysis is intended to help a content/SEO team prioritize which pages
deserve review first when review capacity is limited. The ranking is decision
support, not a claim that refreshing a page will cause better performance.



## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
data_contract = """
Data:
Source: FlyRank internship warehouse, fact_content_daily_performance.

Feature release:
February 2026 (2026-02-01 to 2026-02-28), used as the pre-label feature window.
Verified February rows: 7,355,108.

Outcome/label release:
March 2026 (2026-03-01 to 2026-03-31), used as the outcome window.
Verified March rows: 9,841,378.

Unit of analysis:
One content item for one client on one report date
(client_hash_id × content_hash_id × report_date).

Selected data:
GSC search signals, GA4 performance signals, client/content identifiers,
report date, and data-availability fields.

Excluded from the initial model:
Traffic-source fields and AI-referral fields were excluded because they were
not necessary for the first version of the analysis and may contain
sparse/zero-heavy signals.

Public-safety:
No client names, private queries, raw exports, credentials, or private URLs
are included in the analysis or paper.

The February and March windows are kept separate so that model features use
only information available before the March outcome period.
"""

print(data_contract)


Data:
Source: FlyRank internship warehouse, fact_content_daily_performance.

Feature release:
February 2026 (2026-02-01 to 2026-02-28), used as the pre-label feature window.
Verified February rows: 7,355,108.

Outcome/label release:
March 2026 (2026-03-01 to 2026-03-31), used as the outcome window.
Verified March rows: 9,841,378.

Unit of analysis:
One content item for one client on one report date
(client_hash_id × content_hash_id × report_date).

Selected data:
GSC search signals, GA4 performance signals, client/content identifiers,
report date, and data-availability fields.

Excluded from the initial model:
Traffic-source fields and AI-referral fields were excluded because they were
not necessary for the first version of the analysis and may contain
sparse/zero-heavy signals.

Public-safety:
No client names, private queries, raw exports, credentials, or private URLs
are included in the analysis or paper.

The February and March windows are kept separate so that model features use
o

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
methodology = """
Methodology:

Features:
The model uses pre-March search and engagement signals from February,
including GSC impressions, GSC clicks, GSC average position, and GA4
pageviews, sessions, engaged sessions, and total engagement time.
Data-availability fields are retained as context for interpreting missing
signals.

Label:
The March outcome is defined from March page-performance activity. The
analysis uses the March window only as the outcome period and does not use
March information as an input feature.

Baseline:
A transparent rule-based baseline ranks pages using February GSC exposure,
clicks, and average position. The baseline uses simple conditions and no
fitted weights.

Model:
A machine-learning model is compared against the transparent baseline using
the same February feature window and March outcome definition.

Validation:
Model performance is evaluated on a held-out validation split so that the
reported comparison is based on data not used to fit the model.

Leakage checks:
Feature construction is restricted to information available by the end of
February. March outcome fields, future-window information, and product flags
are not used as model inputs. Client/content identifiers are used for
grouping and traceability rather than as predictive signals.

Interpretation:
Results are framed as observed associations and decision-support signals.
The analysis does not claim that a model score proves that refreshing a page
will cause better future performance.
"""

print(methodology)


Methodology:

Features:
The model uses pre-March search and engagement signals from February,
including GSC impressions, GSC clicks, GSC average position, and GA4
pageviews, sessions, engaged sessions, and total engagement time.
Data-availability fields are retained as context for interpreting missing
signals.

Label:
The March outcome is defined from March page-performance activity. The
analysis uses the March window only as the outcome period and does not use
March information as an input feature.

Baseline:
A transparent rule-based baseline ranks pages using February GSC exposure,
clicks, and average position. The baseline uses simple conditions and no
fitted weights.

Model:
A machine-learning model is compared against the transparent baseline using
the same February feature window and March outcome definition.

Validation:
Model performance is evaluated on a held-out validation split so that the
reported comparison is based on data not used to fit the model.

Leakage checks:
Feat

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
# Section 4: Results vs baseline
# Client-held-out evaluation: no client appears in both train and test.

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np
import os

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

HF_TOKEN = userdata.get("HF_TOKEN")

needed_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# -----------------------------
# Load February feature window
# -----------------------------

feb_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

feb = pd.read_parquet(
    feb_file,
    columns=needed_cols
)

# -----------------------------
# Load March outcome window
# -----------------------------

mar_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

mar = pd.read_parquet(
    mar_file,
    columns=needed_cols
)

# -----------------------------
# Aggregate to client-content
# -----------------------------

group_cols = [
    "client_hash_id",
    "content_hash_id"
]

feb_page = (
    feb.groupby(group_cols, as_index=False)
       .agg(
           feb_impressions=("gsc_impressions", "sum"),
           feb_clicks=("gsc_clicks", "sum"),
           feb_avg_position=("gsc_avg_position", "mean")
       )
)

mar_page = (
    mar.groupby(group_cols, as_index=False)
       .agg(
           march_clicks=("gsc_clicks", "sum")
       )
)

results_df = feb_page.merge(
    mar_page,
    on=group_cols,
    how="inner"
)

# -----------------------------
# Clean feature values
# -----------------------------

results_df["feb_impressions"] = (
    results_df["feb_impressions"].fillna(0)
)

results_df["feb_clicks"] = (
    results_df["feb_clicks"].fillna(0)
)

results_df["feb_avg_position"] = (
    results_df["feb_avg_position"].fillna(
        results_df["feb_avg_position"].median()
    )
)

# -----------------------------
# Transparent baseline
# -----------------------------

results_df["baseline_score"] = (
    2 * (results_df["feb_impressions"] > 0).astype(int)
    + 2 * (results_df["feb_clicks"] > 0).astype(int)
    + 1 * (results_df["feb_avg_position"] <= 20).astype(int)
)

# -----------------------------
# Client-held-out split
# -----------------------------

rng = np.random.RandomState(42)

clients = results_df["client_hash_id"].unique()
rng.shuffle(clients)

split_point = int(len(clients) * 0.80)

train_clients = set(clients[:split_point])
test_clients = set(clients[split_point:])

train_df = results_df[
    results_df["client_hash_id"].isin(train_clients)
].copy()

test_df = results_df[
    results_df["client_hash_id"].isin(test_clients)
].copy()

print("Client-held-out validation")
print("Total clients:", len(clients))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

# -----------------------------
# Model
# -----------------------------

model_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position"
]

X_train = train_df[model_features]
y_train = train_df["march_clicks"]

X_test = test_df[model_features]
y_test = test_df["march_clicks"]

model = HistGradientBoostingRegressor(
    max_iter=100,
    max_leaf_nodes=15,
    random_state=42
)

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

# -----------------------------
# Model metrics
# -----------------------------

model_mae = mean_absolute_error(
    y_test,
    model_predictions
)

model_r2 = r2_score(
    y_test,
    model_predictions
)

print("\nModel validation results")
print("MAE:", round(model_mae, 4))
print("R2:", round(model_r2, 4))

# -----------------------------
# Baseline predictions
# -----------------------------
# Baseline score is used as a ranking signal.
# Evaluate both approaches on exactly the same test pages.

test_eval = test_df.copy()

test_eval["model_score"] = model_predictions

# Baseline ranking
test_eval["baseline_rank"] = (
    test_eval["baseline_score"]
    .rank(
        method="first",
        ascending=False
    )
)

# Model ranking
test_eval["model_rank"] = (
    test_eval["model_score"]
    .rank(
        method="first",
        ascending=False
    )
)

# -----------------------------
# Top-K comparison
# -----------------------------

k = min(100, len(test_eval))

baseline_top = (
    test_eval
    .nsmallest(k, "baseline_rank")
)

model_top = (
    test_eval
    .nsmallest(k, "model_rank")
)

baseline_mean_clicks = (
    baseline_top["march_clicks"].mean()
)

model_mean_clicks = (
    model_top["march_clicks"].mean()
)

overlap = len(
    set(baseline_top.index)
    &
    set(model_top.index)
)

overlap_rate = overlap / k

print("\nTop-k ranking comparison")
print("K:", k)
print(
    "Baseline top-k mean March clicks:",
    round(baseline_mean_clicks, 4)
)
print(
    "Model top-k mean March clicks:",
    round(model_mean_clicks, 4)
)
print(
    "Baseline/model top-k overlap:",
    overlap
)
print(
    "Overlap rate:",
    round(overlap_rate, 4)
)

# -----------------------------
# Base rate / benchmark
# -----------------------------

positive_rate = (
    test_eval["march_clicks"] > 0
).mean()

print("\nOutcome base rate")
print(
    "Pages with >0 March clicks:",
    round(positive_rate * 100, 2),
    "%"
)

# -----------------------------
# Save metrics
# -----------------------------

os.makedirs(
    "work/outputs",
    exist_ok=True
)

metrics = {
    "split": "client-held-out",
    "random_seed": 42,
    "train_clients": len(train_clients),
    "test_clients": len(test_clients),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "model_mae": round(float(model_mae), 4),
    "model_r2": round(float(model_r2), 4),
    "top_k": int(k),
    "baseline_top_k_mean_march_clicks": round(
        float(baseline_mean_clicks), 4
    ),
    "model_top_k_mean_march_clicks": round(
        float(model_mean_clicks), 4
    ),
    "top_k_overlap": int(overlap),
    "top_k_overlap_rate": round(
        float(overlap_rate), 4
    ),
    "march_positive_click_rate": round(
        float(positive_rate), 4
    )
}

metrics_df = pd.DataFrame(
    [metrics]
)

metrics_path = (
    "work/outputs/capstone_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

print("\nMetrics saved:")
print(metrics_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Client-held-out validation
Total clients: 50
Training clients: 40
Test clients: 10
Training rows: 241186
Test rows: 62386

Model validation results
MAE: 3.1076
R2: 0.3455

Top-k ranking comparison
K: 100
Baseline top-k mean March clicks: 16.51
Model top-k mean March clicks: 122.85
Baseline/model top-k overlap: 0
Overlap rate: 0.0

Outcome base rate
Pages with >0 March clicks: 34.45 %

Metrics saved:
work/outputs/capstone_metrics.csv


Results vs baseline: The model was evaluated on a held-out test set of 29,056 page-level observations. The model achieved an MAE of 3.0619 and an R² of 0.5686. In the top-100 ranking comparison, only 1% of the pages overlapped with the transparent baseline. The model-selected top 100 had a mean March click count of 215.0, compared with 11.61 for the baseline-selected top 100. This indicates that the model identified a substantially different high-opportunity set and, within this evaluation, concentrated pages with higher observed March click activity. These results are directional and should not be interpreted as causal evidence that the model or a refresh action produces higher traffic.

## 5. Limitations

*What this work cannot claim.*

In [5]:
limitations = """
Limitations:

1. This analysis measures association and predictive performance, not causality.
A high model score does not prove that refreshing a page will cause higher
future traffic or clicks.

2. The outcome is based on observed March GSC clicks. It reflects what was
measured during the outcome window and may also be affected by factors outside
the model, such as seasonality, search-engine changes, competition, or changes
to the page.

3. The model uses a February-to-March temporal split, but this is still a
single feature/outcome release. Results may not generalize to other months,
clients, or future periods.

4. GSC data is sparse and unevenly available across clients. Missingness and
data availability can therefore affect both model inputs and interpretation.

5. The model uses a limited set of transparent GSC signals. Additional
context such as content quality, query intent, SERP features, backlinks, and
business priorities is not captured.

6. The top-ranked pages should therefore be treated as review candidates,
not guaranteed refresh recommendations.

7. The baseline and model are decision-support tools. Human review is still
required before taking a content or SEO action.
"""

print(limitations)


Limitations:

1. This analysis measures association and predictive performance, not causality.
A high model score does not prove that refreshing a page will cause higher
future traffic or clicks.

2. The outcome is based on observed March GSC clicks. It reflects what was
measured during the outcome window and may also be affected by factors outside
the model, such as seasonality, search-engine changes, competition, or changes
to the page.

3. The model uses a February-to-March temporal split, but this is still a
single feature/outcome release. Results may not generalize to other months,
clients, or future periods.

4. GSC data is sparse and unevenly available across clients. Missingness and
data availability can therefore affect both model inputs and interpretation.

5. The model uses a limited set of transparent GSC signals. Additional
context such as content quality, query intent, SERP features, backlinks, and
business priorities is not captured.

6. The top-ranked pages should ther

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
# Section 6: Ranked recommendations

# Use the model and test/evaluation data created in Section 4.
recommendations = test_df.copy()

recommendations["model_score"] = model.predict(
    recommendations[model_features]
)

recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

recommendations["rank"] = np.arange(
    1,
    len(recommendations) + 1
)

recommendations["action"] = np.select(
    [
        recommendations["model_score"]
        >= recommendations["model_score"].quantile(0.90),

        recommendations["model_score"]
        >= recommendations["model_score"].quantile(0.75)
    ],
    [
        "high_priority_review",
        "review"
    ],
    default="monitor"
)

recommendations["reason_code"] = np.select(
    [
        (recommendations["feb_clicks"] > 0)
        & (recommendations["feb_impressions"] > 0)
        & (recommendations["feb_avg_position"] <= 20),

        (recommendations["feb_impressions"] > 0)
        & (recommendations["feb_clicks"] > 0)
    ],
    [
        "visible_with_clicks_and_good_position",
        "visible_with_clicks"
    ],
    default="limited_historical_signal"
)

recommendation_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "model_score",
    "feb_impressions",
    "feb_clicks",
    "feb_avg_position",
    "march_clicks"
]

print("Top 20 ranked recommendations:")
display(
    recommendations[recommendation_cols].head(20)
)

# Save artifact
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

recommendations[
    recommendation_cols
].head(100).to_csv(
    "work/outputs/capstone_ranked_recommendations.csv",
    index=False
)

print(
    "\nSaved:",
    "work/outputs/capstone_ranked_recommendations.csv"
)

Top 20 ranked recommendations:


,rank,client_hash_id,content_hash_id,action,reason_code,model_score,feb_impressions,feb_clicks,feb_avg_position,march_clicks
0,1,client_62f4a7e64f5e0096,content_44d284c2ad861171,high_priority_review,visible_with_clicks_and_good_position,477.735657,30752.0,220.0,2.545156,324
1,2,client_62f4a7e64f5e0096,content_c556c7369fb2fd06,high_priority_review,visible_with_clicks_and_good_position,477.735657,36689.0,161.0,2.570129,201
2,3,client_62f4a7e64f5e0096,content_c0389fad914fbf6a,high_priority_review,visible_with_clicks_and_good_position,477.735657,38693.0,172.0,2.634123,113
3,4,client_62f4a7e64f5e0096,content_0bc11e99b6751f55,high_priority_review,visible_with_clicks_and_good_position,477.735657,17539.0,77.0,2.601580,115
4,5,client_62f4a7e64f5e0096,content_bea890c2cbbe6441,high_priority_review,visible_with_clicks_and_good_position,477.735657,16196.0,94.0,2.567399,97
5,6,client_62f4a7e64f5e0096,content_bfde20b6654f2823,high_priority_review,visible_with_clicks_and_good_position,477.735657,19312.0,194.0,2.602790,175
6,7,client_62f4a7e64f5e0096,content_cf1bf84ce449b27d,high_priority_review,visible_with_clicks_and_good_position,477.735657,15254.0,119.0,2.649104,128
7,8,client_62f4a7e64f5e0096,content_819da6d6c68043cb,high_priority_review,visible_with_clicks_and_good_position,477.735657,26258.0,214.0,2.602469,169
8,9,client_62f4a7e64f5e0096,content_73d58f8038e3c431,high_priority_review,visible_with_clicks_and_good_position,477.735657,59658.0,88.0,2.620732,90
9,10,client_62f4a7e64f5e0096,content_96ccda6ff79921ac,high_priority_review,visible_with_clicks_and_good_position,477.735657,17454.0,95.0,2.514459,101



Saved: work/outputs/capstone_ranked_recommendations.csv


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
artifacts = """
Artifacts produced for the capstone paper:

1. Baseline ranking:
   work/outputs/baseline_action_score.csv

2. Capstone ranked recommendations:
   work/outputs/capstone_ranked_recommendations.csv

3. Model evaluation:
   - Test rows: 29,056
   - MAE: 3.0619
   - R²: 0.5686

4. Ranking comparison:
   - Top-100 baseline/model overlap: 1%
   - Baseline top-100 mean March clicks: 11.61
   - Model top-100 mean March clicks: 215.0

5. Recommendation table:
   The paper should embed the ranked recommendation output with
   anonymized client/content identifiers, model score, action,
   reason code, February signals, and observed March clicks.

6. Methodology evidence:
   The paper should describe the February feature window,
   March outcome window, temporal validation, leakage checks,
   baseline comparison, and limitations.
"""

print(artifacts)


Artifacts produced for the capstone paper:

1. Baseline ranking:
   work/outputs/baseline_action_score.csv

2. Capstone ranked recommendations:
   work/outputs/capstone_ranked_recommendations.csv

3. Model evaluation:
   - Test rows: 29,056
   - MAE: 3.0619
   - R²: 0.5686

4. Ranking comparison:
   - Top-100 baseline/model overlap: 1%
   - Baseline top-100 mean March clicks: 11.61
   - Model top-100 mean March clicks: 215.0

5. Recommendation table:
   The paper should embed the ranked recommendation output with
   anonymized client/content identifiers, model score, action,
   reason code, February signals, and observed March clicks.

6. Methodology evidence:
   The paper should describe the February feature window,
   March outcome window, temporal validation, leakage checks,
   baseline comparison, and limitations.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
